# 10. Prediction Demo

In this notebook I test my saved salary prediction model with manual inputs.

This is not for training. This is for checking whether the saved model and metadata work correctly before I build a Streamlit app.

## 1. Import Libraries

In [7]:
from pathlib import Path
import json

import joblib
import pandas as pd

pd.set_option("display.max_columns", None)

## 2. Load Saved Model And Metadata

I load the saved Ridge pipeline and the metadata that contains my input columns and rare-category rules.

In [8]:
models_dir = Path("../models")

model_path = models_dir / "salary_prediction_ridge_pipeline.joblib"
metadata_path = models_dir / "salary_prediction_metadata.json"

loaded_model = joblib.load(model_path)

with open(metadata_path, "r") as file:
    loaded_metadata = json.load(file)

print("Model and metadata loaded successfully.")
print(f"Model name: {loaded_metadata['model_name']}")

Model and metadata loaded successfully.
Model name: salary_prediction_ridge_pipeline


## 3. Create Rare Category Mapping Function

Before prediction, I need to apply the same rare-category grouping that I used during final model training.

If a new input value is not in the kept category list, I convert it to `Other`.

In [9]:
def apply_rare_category_mapping(input_data, metadata):
    input_data = input_data.copy()

    for column in metadata["rare_category_columns"]:
        kept_categories = metadata["rare_category_mapping"][column]["kept_categories"]

        input_data[column] = input_data[column].where(
            input_data[column].isin(kept_categories),
            "Other"
        )

    return input_data

## 4. Create Manual Prediction Inputs

I create a few fake salary profiles to test the model like a real user would.

I include one rare job title to confirm that the mapping changes it to `Other`.

In [10]:
manual_inputs = pd.DataFrame([
    {
        "work_year": 2022,
        "experience_level": "SE",
        "employment_type": "FT",
        "job_title": "Data Scientist",
        "employee_residence": "US",
        "remote_ratio": 100,
        "company_location": "US",
        "company_size": "M"
    },
    {
        "work_year": 2022,
        "experience_level": "EN",
        "employment_type": "FT",
        "job_title": "Data Analyst",
        "employee_residence": "IN",
        "remote_ratio": 50,
        "company_location": "IN",
        "company_size": "S"
    },
    {
        "work_year": 2022,
        "experience_level": "MI",
        "employment_type": "FT",
        "job_title": "NLP Engineer",
        "employee_residence": "DE",
        "remote_ratio": 100,
        "company_location": "DE",
        "company_size": "M"
    },
    {
        "work_year": 2022,
        "experience_level": "EX",
        "employment_type": "FT",
        "job_title": "Data Science Manager",
        "employee_residence": "GB",
        "remote_ratio": 0,
        "company_location": "GB",
        "company_size": "L"
    }
])

manual_inputs = manual_inputs[loaded_metadata["input_columns"]]

manual_inputs

,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size
0,2022,SE,FT,Data Scientist,US,100,US,M
1,2022,EN,FT,Data Analyst,IN,50,IN,S
2,2022,MI,FT,NLP Engineer,DE,100,DE,M
3,2022,EX,FT,Data Science Manager,GB,0,GB,L


## 5. Apply Rare Category Mapping

Here I check how the manual inputs look after applying the same category grouping rules.

In [11]:
manual_inputs_grouped = apply_rare_category_mapping(
    manual_inputs,
    loaded_metadata
)

manual_inputs_grouped

,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size
0,2022,SE,FT,Data Scientist,US,100,US,M
1,2022,EN,FT,Data Analyst,IN,50,IN,S
2,2022,MI,FT,Other,DE,100,DE,M
3,2022,EX,FT,Data Science Manager,GB,0,GB,L


## 6. Predict Salaries

Now I use the loaded model to predict salaries for the manual examples.

In [12]:
predicted_salaries = loaded_model.predict(manual_inputs_grouped)

prediction_results = manual_inputs.copy()
prediction_results["mapped_job_title"] = manual_inputs_grouped["job_title"]
prediction_results["mapped_employee_residence"] = manual_inputs_grouped["employee_residence"]
prediction_results["mapped_company_location"] = manual_inputs_grouped["company_location"]
prediction_results["predicted_salary_usd"] = predicted_salaries.round(2)

prediction_results

,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size,mapped_job_title,mapped_employee_residence,mapped_company_location,predicted_salary_usd
0,2022,SE,FT,Data Scientist,US,100,US,M,Data Scientist,US,US,153896.18
1,2022,EN,FT,Data Analyst,IN,50,IN,S,Data Analyst,IN,IN,-10081.13
2,2022,MI,FT,NLP Engineer,DE,100,DE,M,Other,DE,DE,79931.63
3,2022,EX,FT,Data Science Manager,GB,0,GB,L,Data Science Manager,GB,GB,145500.27


## Prediction Demo Observation

If this notebook runs without errors, then my saved model is ready to be used in a Streamlit app.

Main things I check here:

- the model loads correctly
- metadata loads correctly
- rare categories are mapped to `Other`
- predictions are numeric and in a reasonable salary range

## 7. Clip Unrealistic Negative Predictions

Linear/Ridge Regression can sometimes predict negative values because the model output is not naturally bounded.

Salary cannot be negative, so for demo/app display I clip predictions below 0 to 0.

This does not change the trained model. It only changes how I display predictions.

In [14]:
negative_predictions = prediction_results[
    prediction_results["predicted_salary_usd"] < 0
]

negative_predictions

,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size,mapped_job_title,mapped_employee_residence,mapped_company_location,predicted_salary_usd
1,2022,EN,FT,Data Analyst,IN,50,IN,S,Data Analyst,IN,IN,-10081.13


## Negative Prediction Observation

The model can produce negative values for some low-salary profiles because Ridge Regression is still a linear model.

For app display, I will use `predicted_salary_usd_clipped`.

This is a practical display fix, not a model training fix.